# Milestone 1 — A Variational Autoencoder on 2D Data

**Course:** Noise → Masterpiece (Build Stable Diffusion from First Principles) — Week 3

This notebook builds a VAE from scratch on 2D synthetic data (two interleaving moons), trains it,
and shows it can **reconstruct**, organize a **latent space**, and **generate new points from pure noise**.

**The loss (ELBO), in one line.** We maximize `log p(x)`, but it's an intractable integral over all latent
codes. Introducing an encoder `q(z|x)` gives a tractable lower bound — the ELBO — whose negative is our loss:

$$\mathcal{L} = \underbrace{\|x-\hat{x}\|^2}_{\text{reconstruction}} \;+\; \underbrace{-\tfrac12\sum\left(1+\log\sigma^2-\mu^2-\sigma^2\right)}_{\text{KL}(q(z|x)\,\|\,N(0,I))}$$

The reconstruction term forces the decoder to rebuild the input; the KL term keeps the latent space close to
`N(0, I)` so it stays smooth and sampleable.

**Deliverables (mapped at the bottom):** (1) a working VAE class, (2) decreasing loss,
(3) reconstruction plot, (4) latent-space plot, (5) generated samples from noise.


## Setup

In [ ]:
import torch
import numpy as np
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import make_moons, make_circles, make_blobs
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

## Step 1 — Synthetic 2D data

We use 2D data because it trains in seconds on CPU and we can *see* whether it works.
The two moons are two interleaving crescents — a good test of whether the model can capture
**two separate modes**.

We standardize the data (zero mean, unit variance per axis). This isn't cosmetic: the prior is
`N(0, I)`, so matching the data's scale to the prior makes generation noticeably cleaner.

In [ ]:
def get_data(name="moons", n=3000, batch_size=256):
    if name == "moons":
        X, y = make_moons(n_samples=n, noise=0.05)
    elif name == "circles":
        X, y = make_circles(n_samples=n, noise=0.04, factor=0.5)
    elif name == "blobs":
        X, y = make_blobs(n_samples=n, centers=3, cluster_std=0.6)

    # standardize -> prior is N(0, I), so zero-mean/unit-var data generates more cleanly
    X = (X - X.mean(0)) / X.std(0)

    X_t = torch.FloatTensor(X)
    y_t = torch.LongTensor(y)          # labels kept ONLY for coloring the latent plot
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=True)
    return loader, X_t, y_t

loader, X, y = get_data("moons")
print("data shape:", X.shape)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], s=8, c=y, cmap="coolwarm", alpha=0.6)
plt.title("Training data: two moons (color = class, for reference only)")
plt.axis("equal"); plt.show()

## Step 2 — The VAE (required baseline: `LinearVAE`)

The encoder outputs **two** numbers per latent dim: a mean `μ` and a log-variance `log σ²` — i.e. a little
Gaussian, not a point. We output `log σ²` instead of `σ²` because variance must be positive, but a network
can emit any real number; `σ² = exp(log σ²)` is then always positive with no constraint to enforce.

For 2D data the encoder and decoder are single linear layers.

In [ ]:
class LinearVAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=2):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim * 2)   # -> [mu | logvar]
        self.decoder = nn.Linear(latent_dim, input_dim)

    def encode(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=-1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        # z = mu + sigma * eps, eps ~ N(0, I)  -- randomness lives in eps, so gradients flow through mu, sigma
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

## Step 3 — The ELBO loss

`reduction='sum'` on both terms keeps them on the same scale (averaging one and summing the other lets one
silently dominate). `beta` is the KL weight: `beta=1` is the exact ELBO; we'll use it below and revisit it later.

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon = nn.functional.mse_loss(recon_x, x, reduction="sum")          # -E[log p(x|z)] for Gaussian likelihood
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())         # KL(q(z|x) || N(0, I)), closed form
    return recon + beta * kl, recon, kl

## Step 4 — Training loop

In [ ]:
def train_vae(model, loader, epochs=300, lr=1e-3, beta=1.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"loss": [], "recon": [], "kl": []}
    for ep in range(epochs):
        tl = tr = tk = 0.0
        for (x,) in loader:
            recon, mu, logvar = model(x)
            loss, rl, kl = vae_loss(recon, x, mu, logvar, beta)
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item(); tr += rl.item(); tk += kl.item()
        hist["loss"].append(tl); hist["recon"].append(tr); hist["kl"].append(tk)
        if ep % 50 == 0:
            print(f"epoch {ep:3d} | loss {tl:8.1f} | recon {tr:8.1f} | kl {tk:7.1f}")
    return hist

In [ ]:
linear_vae = LinearVAE()
hist_lin = train_vae(linear_vae, loader, epochs=300, beta=1.0)

## Step 5 — Visualize

Four plots: training loss, reconstructions (red should sit on blue), the latent space (colored by class to
see structure), and samples generated by feeding pure `N(0, I)` noise to the decoder.

In [ ]:
def show_results(model, hist, X, y, title, beta=1.0):
    model.eval()
    with torch.no_grad():
        recon, _, _ = model(X)
        mu, _ = model.encode(X)
        gen = model.decode(torch.randn(2000, 2))   # generate from pure noise

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))

    ax[0].plot(hist["loss"], label="total")
    ax[0].plot(hist["recon"], label="recon")
    ax[0].plot([beta * k for k in hist["kl"]], label=f"{beta:g}·KL")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()

    ax[1].scatter(X[:400, 0], X[:400, 1], s=8, c="steelblue", alpha=0.5, label="original")
    ax[1].scatter(recon[:400, 0], recon[:400, 1], s=8, c="crimson", alpha=0.5, label="reconstructed")
    ax[1].set_title("Reconstruction"); ax[1].legend(); ax[1].axis("equal")

    ax[2].scatter(mu[:, 0], mu[:, 1], s=8, c=y, cmap="coolwarm", alpha=0.6)
    ax[2].set_title("Latent space (color = class)"); ax[2].set_xlabel("z1"); ax[2].set_ylabel("z2")

    ax[3].scatter(X[:600, 0], X[:600, 1], s=6, c="lightgray", alpha=0.3, label="real")
    ax[3].scatter(gen[:, 0], gen[:, 1], s=6, c="seagreen", alpha=0.4, label="generated")
    ax[3].set_title("Generated from N(0, I)"); ax[3].legend(); ax[3].axis("equal")

    fig.suptitle(title, fontsize=14); fig.tight_layout(); plt.show()

show_results(linear_vae, hist_lin, X, y, "LinearVAE (single linear layers) — baseline", beta=1.0)

### What the LinearVAE shows (and why generation looks wrong)

Reconstruction is a blurry linear smear and **generation collapses to a single elliptical blob** — it does not
look like two moons. This isn't a bug; it's the math. The decoder is a single linear map, and **a linear map of
a Gaussian is always a Gaussian.** Feed `N(0, I)` through `Linear(z)` and you can only ever get one elliptical
cloud — never two disjoint curved crescents. (The latent plot still shows real structure: even a linear encoder
separates the two classes.)

To generate a curved, multi-mode manifold we need a **nonlinear** decoder.

## Step 6 — A deeper VAE (nonlinear decoder)

Same VAE skeleton — encoder → reparameterize → decoder, same ELBO loss — but the linear layers become small
MLPs with ReLU. This is the only change needed to capture a nonlinear manifold.

In [ ]:
class MLPVAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=2, h=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, latent_dim * 2),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, input_dim),
        )

    def encode(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=-1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

We train with `beta = 0.15` (a slightly down-weighted KL). With the full `beta=1` ELBO, the KL term is so
strong it squashes the second latent dimension and both crescents collapse onto one curve. Easing the KL
pressure lets the latent keep enough information to bend into **both** moons. This `beta` knob is exactly the
β-VAE idea from the bonus section — turning KL up enforces a tidier latent at the cost of reconstruction, turning
it down does the reverse.

In [ ]:
mlp_vae = MLPVAE()
hist_mlp = train_vae(mlp_vae, loader, epochs=600, beta=0.15)
show_results(mlp_vae, hist_mlp, X, y, "MLP VAE (nonlinear) — captures the manifold", beta=0.15)

### Result

Reconstruction now lands on **both** crescents, the latent space cleanly separates the two modes, and generation
from pure `N(0, I)` noise reproduces both moons. The light scatter between the crescents is the VAE faithfully
interpolating across a *continuous* latent space — exactly the smoothness the KL term buys us, and the property
that makes latent interpolation (next cell) work.

## Bonus — Latent-space interpolation

Pick one point from each moon, encode both to their `μ` codes, walk a straight line between them in latent
space, and decode each step. A smooth path through valid-looking points is the signature of a well-formed VAE
latent space.

In [ ]:
mlp_vae.eval()
with torch.no_grad():
    idx0 = (y == 0).nonzero()[0].item()
    idx1 = (y == 1).nonzero()[0].item()
    mu0, _ = mlp_vae.encode(X[idx0:idx0+1])
    mu1, _ = mlp_vae.encode(X[idx1:idx1+1])
    ts = torch.linspace(0, 1, 12).unsqueeze(1)
    path_z = (1 - ts) * mu0 + ts * mu1            # interpolate in latent space
    path_x = mlp_vae.decode(path_z)

plt.figure(figsize=(6, 6))
plt.scatter(X[:600, 0], X[:600, 1], s=6, c="lightgray", alpha=0.3)
plt.plot(path_x[:, 0], path_x[:, 1], "-o", c="darkorange", ms=6)
plt.title("Latent interpolation: one moon → the other")
plt.axis("equal"); plt.show()

## Deliverable checklist

| # | Deliverable | Where |
|---|---|---|
| 1 | A working VAE class | `LinearVAE` (Step 2) and `MLPVAE` (Step 6) |
| 2 | Training loss decreasing | loss curves in both `show_results` panels |
| 3 | Reconstruction plot | panel 2 of `show_results` (MLP: red on both moons) |
| 4 | Latent-space visualization | panel 3 (two modes separated) |
| 5 | Generated samples from pure noise | panel 4 (MLP reproduces both moons) |

**One-line takeaway:** a VAE = encoder (data → Gaussian) + reparameterization (sample differentiably) + decoder
(code → data), trained on reconstruction + KL. The KL term is what makes the latent space smooth enough to
sample and interpolate — and a nonlinear decoder is what lets it model a curved, multi-mode manifold.